In [2]:
# Import packages and initialize Earth Engine
import ee
import geemap
import pandas as pd
import numpy as np
import os
import seaborn as sns
import cmocean

ee.Authenticate()
ee.Initialize(project='ee-ivanburgov666')

In [3]:
# Choose whether to create rasters of the acquisition time before or during the orbital drift period

answer = input("Type 'before' or 'during' to create rasters of the acquisition time before or during the orbital drift period.")
if answer == 'before':
    date = '2019-06-01'
    print('Creating raster of acquisition time -BEFORE- the orbital drift period.')
elif answer == 'during':
    date = '2025-06-01'
    print('Creating raster of acquisition time -DURING- the orbital drift period.')
else: print("Invalid input. Please type 'before' or 'during'.")

Creating raster of acquisition time -BEFORE- the orbital drift period.


In [ ]:
# Load mask and geometry

greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)
greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])

In [ ]:
# Create collections and create average images as a stack 

def time_modis(image):
    'Terra Day band selection and conversion'
    overfly_D = image.select('Day_view_time').multiply(0.1).rename('Dtime')
    overfly_N = image.select('Night_view_time').multiply(0.1).rename('Ntime')
    return image.addBands(overfly_D).addBands(overfly_N).select(['Dtime', 'Ntime'])

MOD11A1 = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['Day_view_time', 'Night_view_time'])
    .filterDate(date, ee.Date(date).advance(1, 'months'))
    .filterBounds(greenland)
    .map(time_modis)
)

MYD11A1 = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['Day_view_time', 'Night_view_time'])
    .filterDate(date, ee.Date(date).advance(1, 'months'))
    .filterBounds(greenland)
    .map(time_modis)
)

MOD_D = MOD11A1.select('Dtime').mean().updateMask(greenlandmask).rename([f'MOD_Dtime_{answer}'])
MOD_N = MOD11A1.select('Ntime').mean().updateMask(greenlandmask).rename([f'MOD_Ntime_{answer}'])
MYD_D = MYD11A1.select('Dtime').mean().updateMask(greenlandmask).rename([f'MYD_Dtime_{answer}'])
MYD_N = MYD11A1.select('Ntime').mean().updateMask(greenlandmask).rename([f'MYD_Ntime_{answer}'])

stack = MOD_D.addBands(MOD_N).addBands(MYD_D).addBands(MYD_N)

In [ ]:
# Export the stacked image to Google Drive

task = ee.batch.Export.image.toDrive(
    image=stack,
    description=f'Acquisition_time_{answer}_1month',
    folder='GEMLST/OrbitalDrift',
    fileNamePrefix=f'Acquisition_time_{answer}_1month',
    region=greenland,
    scale=1000,
    crs='EPSG:3413',
    maxPixels=1e13
)

task.start()